In [ ]:
import os
import sqlite3
import pandas as pd
from datetime import datetime
import duckdb
from dotenv import load_dotenv

load_dotenv("/home/westerd/_/research_projects/tedla-hypertension/src/nlp_method/.env")

note_id_to_note_groups_table_path = os.getenv("NOTE_ID_TO_NOTE_GROUPS_TABLE")
assert note_id_to_note_groups_table_path is not None

results_db_path = "/home/westerd/_/project_data/tedla-hypertension/results/results.db"

cohort_note_data_table = os.getenv("COHORT_NOTE_DATA_TABLE")
assert cohort_note_data_table is not None

multipart_note_id_map_path = '/home/westerd/_/project_data/tedla-hypertension/source_data/multipart_note_id_mapping'

In [22]:
duckdb_cxn = duckdb.connect()
duckdb_cxn.install_extension("sqlite")
duckdb_cxn.load_extension("sqlite")

In [23]:
duckdb_cxn.execute(f"ATTACH '{results_db_path}' as results_db (TYPE sqlite);")

In [24]:
duckdb_cxn.sql("SHOW TABLES FROM results_db;").df().head(10)

,name
0,results
1,sqlite_sequence


In [25]:
duckdb_cxn.execute("CREATE INDEX IF NOT EXISTS idx_note_id ON results_db.main.results (note_id);")

In [26]:
import sqlite3
from pathlib import Path
from typing import cast
import pandas as pd

def parquet_file_to_sqlite_table(source: str | Path | pd.DataFrame, new_table_name: str):
    
    if isinstance(source, pd.DataFrame) is False:
        df = pd.read_parquet(str(source))
    else:
        df = cast(pd.DataFrame, source)
    results_db_path = "/home/westerd/_/project_data/tedla-hypertension/results/results.db"
    with sqlite3.connect(results_db_path) as sqlite_cxn:
        df.to_sql(new_table_name, sqlite_cxn, if_exists='replace', index=False)

In [27]:
# parquet_file_to_sqlite_table(note_id_to_note_groups_table_path, 'note_id_to_note_groups')

In [28]:
# parquet_file_to_sqlite_table(cohort_note_data_table, 'cohort_note_data')

In [29]:
# parquet_file_to_sqlite_table(multipart_note_id_map_path, 'multipart_note_id_map')

In [30]:
from typing import List
import textwrap


def get_aliased_columns(table_alias: str, columns: List[str]):
    aliased_columns: str = "\n, ".join([f"{table_alias}.{x}" for x in columns])
    return aliased_columns


results_alias = "s"
note_groups_alias = "g"
note_details_alias = "n"
multipart_note_id_map_alias = "m"


results_columns: List[str] = [
    "note_id",
    "window_text",
    "search_term",
    "window_start_char_offset",
    "window_end_char_offset",
    "entity_start_offset",
    "entity_end_offset",
    "is_negated",
    "note_date",
    "visit_start_date",
    "visit_end_date",
]

note_groups_columns = [
    "notes_problem_lists",
    "notes_outpatient",
    "notes_inpatient",
    "notes_emergency_department",
    "notes_ecg_impression",
    "notes_discharge_summary",
    "notes_communication_encounter",
    "notes_admissions",
]

note_details_columns = [
    "note_title",
    "note_source_value",
    "x_source",
    "note_concept_name",
]

mulitpart_note_id_map_columns = ["x_part_num", "x_uniq"]

query = f"""
SELECT
{textwrap.indent(get_aliased_columns(results_alias, results_columns), r'    ')}
{textwrap.indent(", " + get_aliased_columns(note_groups_alias, note_groups_columns), r'    ')}
{textwrap.indent(", " + get_aliased_columns(note_details_alias, note_details_columns), r'    ')}
{textwrap.indent(", " + get_aliased_columns(multipart_note_id_map_alias, mulitpart_note_id_map_columns), r'    ')}
FROM results_db.main.results AS {results_alias}
    inner join read_parquet('{note_id_to_note_groups_table_path}') AS {note_groups_alias}
        ON {results_alias}.note_id = {note_groups_alias}.note_id
    inner join read_parquet('{cohort_note_data_table}') as {note_details_alias}
        on {results_alias}.note_id = {note_details_alias}.note_id
    inner join read_parquet('{multipart_note_id_map_path}') as {multipart_note_id_map_alias}
        on {results_alias}.note_id = {multipart_note_id_map_alias}.note_id
"""
# print(query)

In [31]:
final_results_df = duckdb_cxn.execute(query).fetchdf()

In [32]:
display(duckdb_cxn.execute("select note_id from results_db.main.results").fetchdf().nunique())

note_id    10303849
dtype: int64

In [33]:
display(f"Number of results: {final_results_df['note_id'].nunique()}")

'Number of results: 10303849'

In [35]:
final_results_df[note_groups_columns] = final_results_df[note_groups_columns] == 1
final_results_df[note_details_columns] = final_results_df[note_details_columns].astype(
    "category"
)
final_results_df["search_term"] = final_results_df["search_term"].astype("category")
final_results_df["is_negated"] = final_results_df["is_negated"].astype(bool)
offset_columns = [x for x in final_results_df.columns if x.endswith("_offset")]
dates_columns = ["visit_start_date", "visit_end_date", "note_date"]
for c in dates_columns:
    final_results_df[c] = pd.to_datetime(final_results_df[c], errors="coerce")
    
final_results_df['x_part_num'] = pd.to_numeric(final_results_df['x_part_num'], downcast='integer')
for c in offset_columns:
    final_results_df[c] = pd.to_numeric(final_results_df[c], downcast='integer')

final_results_columns = [
    "note_id",
    "x_uniq",
    "x_part_num",
    "search_term",
    "is_negated",
    "note_title",
    "note_source_value",
    "x_source",
    "note_concept_name",
    "note_date",
    "visit_start_date",
    "visit_end_date",
    "entity_start_offset",
    "entity_end_offset",
    "window_start_char_offset",
    "window_end_char_offset",
    "notes_problem_lists",
    "notes_outpatient",
    "notes_inpatient",
    "notes_emergency_department",
    "notes_ecg_impression",
    "notes_discharge_summary",
    "notes_communication_encounter",
    "notes_admissions",
    "window_text",
]

final_results_df = final_results_df[final_results_columns]

In [38]:
len(final_results_df)

178417295

In [39]:
final_results_df = final_results_df.drop_duplicates()

In [40]:
len(final_results_df)

30760565

In [ ]:
import csv
from pathlib import Path

export_dir = os.getenv("EXPORT_DIR")
assert export_dir is not None
export_dir = Path(export_dir)
export_dir.mkdir(parents=True, exist_ok=True)

display("Building full output...")

import math

rows_per_file = 1_000_000
total_rows = len(final_results_df)
display(f'Total Rows: {total_rows}')
num_files = math.ceil(total_rows / rows_per_file)
display(f'Number of output files to create: {num_files}')

for i in range(num_files):
    display(f"Creating {i} of {num_files}...")
    start = i * rows_per_file
    end = min(start + rows_per_file, total_rows)
    chunk = final_results_df.iloc[start:end]
    chunk.to_csv(export_dir / f"results__note_groups__note_details_{i+1}.csv", index=False, quoting=csv.QUOTE_ALL, quotechar='"', sep=",")

display("Results written to CSV")

'Building full output...'

'Total Rows: 30760565'

'Number of output files to create: 31'

'Creating 0 of 31...'

'Creating 1 of 31...'

...

'Creating 27 of 31...'

'Creating 28 of 31...'

'Creating 29 of 31...'

'Creating 30 of 31...'

'Results written to CSV'

In [46]:
print(export_dir)

/home/westerd/_/project_data/tedla-hypertension/exports


In [49]:
term_present = final_results_df[['search_term', 'window_text']].apply(lambda x: x['search_term'] in x['window_text'], axis=1)


KeyboardInterrupt: 

In [ ]:
term_present.unique()

In [ ]:
description = f"""

Feature Definitions

note_id - the id of the note in Databricks
x_uniq - the original id in the source data
x_part_num - if this is non-null, indicates a multi-part note

  Within the database, some notes exceed a max \"page\" size (~8000 characters)



search_term - the search string
is_negated - whether or not there is an indication of negation
window_text - the surrounding text (20 tokens before and 20 tokens after the search term)

Note Groups - boolean indication of membership to groups defined in NLP method

  - notes_problem_lists
  - notes_outpatient
  - notes_inpatient
  - notes_emergency_department
  - notes_ecg_impression
  - notes_discharge_summary
  - notes_communication_encounter
  - notes_admissions

String Offsets

  - window_start_char_offset
  - window_end_char_offset
  - entity_start_offset
  - entity_end_offset
  
Note Level Data

  - note_title
  - note_source_value
  - x_source
  - note_concept_name
  - note_date
  
Visit Detail Related to Note
  - visit_start_date
  - visit_end_date
"""

In [ ]:
# final_results_columns = [
#     "note_id",
#     "search_term",
#     "is_negated",
#     "note_title",
#     "note_source_value",
#     "x_source",
#     "note_concept_name",
#     "note_date",
#     "visit_start_date",
#     "visit_end_date",
#     "entity_start_offset",
#     "entity_end_offset",
#     "window_start_char_offset",
#     "window_end_char_offset",
#     "notes_problem_lists",
#     "notes_outpatient",
#     "notes_inpatient",
#     "notes_emergency_department",
#     "notes_ecg_impression",
#     "notes_discharge_summary",
#     "notes_communication_encounter",
#     "notes_admissions",
#     "window_text",
# ]

In [ ]:
# display(f"Loading results from sqlite db ({results_db_path})...")

# with sqlite3.connect(results_db_path) as conn:
#     results_df: pd.DataFrame = pd.read_sql_query("SELECT * FROM results", conn)[
#         [
#             "note_id",
#             "window_text",
#             "search_term",
#             "window_start_char_offset",
#             "window_end_char_offset",
#             "entity_start_offset",
#             "entity_end_offset",
#             "is_negated",
#             "note_date",
#             "visit_start_date",
#             "visit_end_date",
#         ]
#     ]

# display(f'{len(results_df)} [{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}]')

# display(f"Results loaded")

In [ ]:
# display("Joining results to note groups by note_id...")

# assert note_id_to_note_groups_table_path is not None

# results_to_note_groups_df: pd.DataFrame = results_df.join(
#     pd.read_parquet(note_id_to_note_groups_table_path).set_index("note_id"),
#     how="inner",
#     on="note_id",
# )[
#     [
#         "note_id",
#         "window_text",
#         "search_term",
#         "notes_problem_lists",
#         "notes_outpatient",
#         "notes_inpatient",
#         "notes_emergency_department",
#         "notes_ecg_impression",
#         "notes_discharge_summary",
#         "notes_communication_encounter",
#         "notes_admissions",
#         "window_start_char_offset",
#         "window_end_char_offset",
#         "entity_start_offset",
#         "entity_end_offset",
#         "is_negated",
#         "note_date",
#         "visit_start_date",
#         "visit_end_date",
#     ]
# ].sort_values(
#     by=["note_id"]
# )

# results_count: int = len(results_df)
# del results_df

# display("Results joined to note groups by note_id (results_to_note_groups_df)")

In [ ]:
# display("Compacting columns...")

# # Compact integer columns
# int_cols = [
#     "note_id",
#     "window_start_char_offset",
#     "window_end_char_offset",
#     "entity_start_offset",
#     "entity_end_offset",
# ]
# for col in int_cols:
#     results_to_note_groups_df[col] = pd.to_numeric(
#         results_to_note_groups_df[col], downcast="integer"
#     )

# # Compact int32 columns (downcast to int8 if possible)
# small_int_cols = [
#     col
#     for col in results_to_note_groups_df.columns
#     if col.startswith("notes_") or "is_negated" == col
# ]
# results_to_note_groups_df[small_int_cols] = results_to_note_groups_df[
#     small_int_cols
# ].astype(bool)

# # Convert date columns to datetime
# date_cols = ["note_date", "visit_start_date", "visit_end_date"]
# for col in date_cols:
#     results_to_note_groups_df[col] = pd.to_datetime(
#         results_to_note_groups_df[col], errors="coerce"
#     )

# # Convert text columns to category if few unique values
# text_cols = ["window_text", "search_term"]
# for col in text_cols:
#     num_unique = results_to_note_groups_df[col].nunique()
#     num_total = len(results_to_note_groups_df)
#     if num_unique / num_total < 0.5:  # adjust threshold as needed
#         results_to_note_groups_df[col] = results_to_note_groups_df[col].astype(
#             "category"
#         )

# results_to_note_groups_count = len(results_to_note_groups_df)

# display("Columns compacted.")

In [ ]:
# display(f"Number rows in results_db: {results_count}")
# display(f"Number rows in results_to_note_groups_df: {results_to_note_groups_count}")

In [ ]:
# results_to_note_groups_df.columns

In [ ]:
# def load_cohort_note_data() -> pd.DataFrame:
#     import pyarrow.dataset as pds

#     cohort_note_data_table = os.getenv("COHORT_NOTE_DATA_TABLE")
#     assert cohort_note_data_table is not None

#     ds = pds.dataset(cohort_note_data_table, format="parquet")
#     cohort_note_columns = [
#         "note_id",
#         "note_title",
#         "note_source_value",
#         "x_source",
#         "note_concept_name",
#     ]
#     note_table = ds.to_table(columns=cohort_note_columns)
#     note_df = note_table.to_pandas()
#     for col in ["note_title", "note_source_value", "x_source", "note_concept_name"]:
#         note_df[col] = note_df[col].astype("category")

#     return note_df

In [ ]:
# notes_df = load_cohort_note_data()
# note_id_count = notes_df["note_id"].nunique()

In [ ]:
# from pathlib import Path

# notes_df = load_cohort_note_data()
# note_id_count = notes_df["note_id"].nunique()

# results__note_groups__note_details_df: pd.DataFrame = results_to_note_groups_df.merge(
#     notes_df, on="note_id", how="inner"
# )[
#     [
#         "note_id",
#         "search_term",
#         "is_negated",
#         "window_text",
#         "notes_problem_lists",
#         "notes_outpatient",
#         "notes_inpatient",
#         "notes_emergency_department",
#         "notes_ecg_impression",
#         "notes_discharge_summary",
#         "notes_communication_encounter",
#         "notes_admissions",
#         "window_start_char_offset",
#         "window_end_char_offset",
#         "entity_start_offset",
#         "entity_end_offset",
#         "note_title",
#         "note_source_value",
#         "x_source",
#         "note_concept_name",
#         "note_date",
#         "visit_start_date",
#         "visit_end_date",
#     ]
# ]

# del notes_df

# results__note_groups__note_details_count = len(results__note_groups__note_details_df)

In [ ]:
# display(f"Note id count in cohort notes set (note_id_count): {note_id_count}")
# display(
#     f'Note id count in results set: {results__note_groups__note_details_df["note_id"].nunique()}'
# )

In [ ]:
# results__note_groups__note_details_df.columns

In [ ]:
# display(
#     f"original results_with_note_groups_df: {len(results__note_groups__note_details_df)}"
# )
# results__note_groups__note_details_df = (
#     results__note_groups__note_details_df.drop_duplicates()
# )
# display(
#     f"deduped results_with_note_groups_df: {len(results__note_groups__note_details_df)}"
# )

In [ ]:
# export_dir = os.getenv("EXPORT_DIR")
# assert export_dir is not None
# export_dir = Path(export_dir)
# export_dir.mkdir(parents=True, exist_ok=True)

# display("Building sample data...")
# results__note_groups__note_details_df.loc[0:2000].to_csv(
#     export_dir / f"sample_results__note_groups__note_details.csv"
# )
# display("Building full output...")
# results__note_groups__note_details_df.to_csv(
#     export_dir / f"results__note_groups__note_details.csv"
# )
# display("Results written to CSV")

In [ ]:
# [x for x in results__note_groups__note_details_df.columns]

In [ ]:
# results__note_groups__note_details_df.head()

In [ ]:
# display(f"Number rows in results_db: {results_count}")
# display(f"Number rows in results_to_note_groups_df: {results_to_note_groups_count}")
# display(
#     f"Number rows results_with_note_groups_count (deduped): {results__note_groups__note_details_count}"
# )

In [ ]:
# # results_with_note_groups_df.head()
# # .replace("notes_", "")
# bool_cols = sorted(
#     [
#         "notes_problem_lists",
#         "notes_outpatient",
#         "notes_inpatient",
#         "notes_emergency_department",
#         "notes_ecg_impression",
#         "notes_discharge_summary",
#         "notes_communication_encounter",
#         "notes_admissions",
#     ]
# )

# # Group and count combinations
# combination_counts = (
#     results__note_groups__note_details_df.groupby(bool_cols)
#     .size()
#     .reset_index(name="count")
# )


# # Create comma-delimited list of columns set to True
# def true_cols(row):
#     return ", ".join([col.replace("notes_", "") for col in bool_cols if row[col]])


# combination_counts["true_columns"] = combination_counts.apply(true_cols, axis=1)

# result: pd.DataFrame = combination_counts[["true_columns", "count"]]

# result.head(100)

In [ ]:
# results__note_groups__note_details_df["is_negated"].value_counts()